In [1]:

import os
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# DAY 15: HEAT-TRANSFER VALIDATION
# --------------------------------------------------

# Find the most recently modified simulation folder
folders = glob.glob("combined_mantle_results*")

if not folders:
    raise FileNotFoundError(
        "Run the combined mantle convection model first."
    )

result_folder = max(folders, key=os.path.getmtime)

files = glob.glob(os.path.join(result_folder, "*.npz"))

if not files:
    raise FileNotFoundError(
        "No .npz simulation files were found."
    )

output_folder = "day15_heat_transfer_validation"
os.makedirs(output_folder, exist_ok=True)

results = []

# --------------------------------------------------
# PROCESS EACH RAYLEIGH NUMBER
# --------------------------------------------------

for filename in files:

    data = np.load(filename)

    required = ["ra", "x", "z", "T", "u", "w"]

    if not all(key in data for key in required):
        print("Skipping file with missing variables:", filename)
        continue

    Ra = float(np.asarray(data["ra"]).item())
    x = data["x"]
    z = data["z"]
    T = data["T"]
    u = data["u"]
    w = data["w"]

    dx = float(np.mean(np.diff(x)))
    dz = float(np.mean(np.diff(z)))

    # Check array dimensions
    if T.shape != u.shape or T.shape != w.shape:
        print("Skipping incompatible array shapes:", filename)
        continue

    # ----------------------------------------------
    # 1. NUSSELT NUMBER AT BOTTOM AND TOP
    # ----------------------------------------------

    # Temperature gradient at bottom boundary
    bottom_gradient = (
        T[1, 1:-1] - T[0, 1:-1]
    ) / dz

    # Temperature gradient at top boundary
    top_gradient = (
        T[-1, 1:-1] - T[-2, 1:-1]
    ) / dz

    Nu_bottom = -np.mean(bottom_gradient)
    Nu_top = -np.mean(top_gradient)

    # Relative mismatch between boundary heat fluxes
    denominator = max(
        abs(Nu_bottom), abs(Nu_top), 1e-12
    )

    mismatch = (
        abs(Nu_bottom - Nu_top) / denominator
    ) * 100

    # ----------------------------------------------
    # 2. TEMPERATURE CHECK
    # ----------------------------------------------

    Tmin = np.min(T)
    Tmax = np.max(T)

    # For the normalized model, boundaries are 0 and 1
    temperature_in_range = (
        Tmin >= -0.05 and Tmax <= 1.05
    )

    # ----------------------------------------------
    # 3. VELOCITY AND DIVERGENCE CHECK
    # ----------------------------------------------

    max_speed = np.max(np.sqrt(u**2 + w**2))

    # Incompressible flow should satisfy:
    # du/dx + dw/dz = 0
    du_dx = np.gradient(u, dx, axis=1)
    dw_dz = np.gradient(w, dz, axis=0)

    divergence = du_dx + dw_dz

    # Ignore the outermost boundary cells
    if min(divergence.shape) > 2:
        interior_divergence = divergence[1:-1, 1:-1]
    else:
        interior_divergence = divergence

    divergence_rms = np.sqrt(
        np.mean(interior_divergence**2)
    )

    # ----------------------------------------------
    # 4. SAVE THE RESULTS
    # ----------------------------------------------

    results.append([
        Ra,
        Nu_bottom,
        Nu_top,
        mismatch,
        Tmin,
        Tmax,
        max_speed,
        divergence_rms,
        temperature_in_range
    ])

    print("\nRayleigh number:", Ra)
    print("Bottom Nu:", Nu_bottom)
    print("Top Nu:", Nu_top)
    print("Heat-flux mismatch (%):", mismatch)
    print("Minimum temperature:", Tmin)
    print("Maximum temperature:", Tmax)
    print("Maximum speed:", max_speed)
    print("RMS divergence:", divergence_rms)

    # ----------------------------------------------
    # 5. TEMPERATURE CONTOUR
    # ----------------------------------------------

    plt.figure(figsize=(7, 4))

    plt.contourf(x, z, T, levels=30, cmap="inferno")
    plt.colorbar(label="Normalized temperature")

    plt.xlabel("Horizontal position, x")
    plt.ylabel("Vertical position, z")
    plt.title(f"Temperature field, Ra = {Ra:g}")

    plt.tight_layout()
    plt.savefig(
        os.path.join(
            output_folder,
            f"day15_temperature_Ra_{Ra:g}.png"
        ),
        dpi=200
    )
    plt.close()

# --------------------------------------------------
# 6. WRITE CSV SUMMARY
# --------------------------------------------------

if not results:
    raise ValueError(
        "No usable simulation datasets were found."
    )

results.sort(key=lambda row: row[0])

headers = [
    "Rayleigh_number",
    "Nu_bottom",
    "Nu_top",
    "Heat_flux_mismatch_percent",
    "Minimum_temperature",
    "Maximum_temperature",
    "Maximum_speed",
    "RMS_divergence",
    "Temperature_in_range"
]

csv_path = os.path.join(
    output_folder,
    "day15_validation_summary.csv"
)

with open(csv_path, "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(headers)
    writer.writerows(results)

# --------------------------------------------------
# 7. PLOT TOP AND BOTTOM NUSSELT NUMBERS
# --------------------------------------------------

Ra_values = [row[0] for row in results]
Nu_bottom_values = [row[1] for row in results]
Nu_top_values = [row[2] for row in results]

plt.figure(figsize=(7, 5))

plt.plot(
    Ra_values, Nu_bottom_values,
    "o-", label="Bottom Nu"
)
plt.plot(
    Ra_values, Nu_top_values,
    "s--", label="Top Nu"
)

plt.axhline(
    1.0, linestyle=":",
    label="Pure conduction (Nu = 1)"
)

plt.xlabel("Rayleigh number")
plt.ylabel("Nusselt number")
plt.title("Day 15: Boundary heat-transfer comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(output_folder, "day15_nusselt_comparison.png"),
    dpi=200
)
plt.close()

# --------------------------------------------------
# 8. PLOT HEAT-FLUX MISMATCH
# --------------------------------------------------

mismatch_values = [row[3] for row in results]

plt.figure(figsize=(7, 5))
plt.plot(Ra_values, mismatch_values, "o-")

plt.xlabel("Rayleigh number")
plt.ylabel("Heat-flux mismatch (%)")
plt.title("Day 15: Top-bottom heat-flux mismatch")
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(output_folder, "day15_heat_flux_mismatch.png"),
    dpi=200
)
plt.close()

# --------------------------------------------------
# 9. FINAL MESSAGE
# --------------------------------------------------

print("\n" + "=" * 45)
print("DAY 15 ANALYSIS COMPLETED")
print("=" * 45)
print("Input folder:", result_folder)
print("Output folder:", output_folder)
print("CSV summary:", csv_path)
print("Plots saved successfully.")



Rayleigh number: 1000.0
Bottom Nu: 5.836153095734757
Top Nu: 7.763581696574158
Heat-flux mismatch (%): 24.826538525252055
Minimum temperature: 0.0
Maximum temperature: 1.0
Maximum speed: 261.5621970066285
RMS divergence: 1.4315010626459518e-13

Rayleigh number: 2500.0
Bottom Nu: 9.746131680905368
Top Nu: 12.454051976431867
Heat-flux mismatch (%): 21.743287250213715
Minimum temperature: 0.0
Maximum temperature: 1.0
Maximum speed: 564.3629566969514
RMS divergence: 3.1221982537956443e-13

Rayleigh number: 500.0
Bottom Nu: 7.0535586228536165
Top Nu: 7.6511121496079335
Heat-flux mismatch (%): 7.8100217990522784
Minimum temperature: 0.0
Maximum temperature: 1.0
Maximum speed: 133.09757056314587
RMS divergence: 7.312926794296519e-14

Rayleigh number: 5000.0
Bottom Nu: 13.92769140082633
Top Nu: 14.250779182114124
Heat-flux mismatch (%): 2.2671587087202627
Minimum temperature: 0.0
Maximum temperature: 1.0
Maximum speed: 1064.8532313417584
RMS divergence: 5.2766168309471e-13

DAY 15 ANALYSIS CO